# LeetCode 3666 — Minimum Operations to Equalize Binary String bằng Python

- Link: https://leetcode.com/problems/minimum-operations-to-equalize-binary-string/
- Độ khó: **Hard**
- Chủ đề: **BFS trên trạng thái rút gọn** (số chữ số 0), chuyển trạng thái thành **khoảng cùng tính chẵn lẻ**, "con trỏ tới phần tử chưa thăm" (DSU nhảy), điều kiện đóng bằng đếm số lần lật

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def minOperations(self, s: str, k: int) -> int:
```

## 1. Mục tiêu học

1. Rút gọn trạng thái: chỉ **số chữ số 0** `z` quan trọng (vị trí cụ thể không ảnh hưởng tập trạng thái đạt được).
2. Từ `z`, một thao tác đưa tới `z + k − 2x` với `x` (số 0 bị lật) trong một khoảng ⇒ đích là một **khoảng** các số **cùng tính chẵn lẻ**.
3. BFS trên `0..n` nhưng mỗi đỉnh có thể có `O(n)` cạnh ⇒ dùng cấu trúc "**phần tử chưa thăm kế tiếp**" để mỗi đỉnh chỉ được duyệt một lần: tổng `O(n α(n))`.
4. Cách nhìn đại số: sau `t` thao tác, mỗi vị trí bị lật `f_i ≤ t` lần, `Σ f_i = t·k`; điều kiện tồn tại cho phép kiểm tra từng `t` trong `O(1)`.
5. Kiểm chứng bằng BFS trên toàn bộ chuỗi `2^n`.

## 2. Đề bài & ví dụ

Mỗi thao tác chọn **đúng `k` chỉ số khác nhau** và lật bit tại đó. Tìm số thao tác ít nhất để mọi ký tự thành `'1'`, hoặc `-1`.

**Ví dụ 1:** `s = "110", k = 1` → `1`.

**Ví dụ 2:** `s = "0101", k = 3` → `2`.

**Ví dụ 3:** `s = "101", k = 2` → `-1`.

**Ràng buộc:** `n ≤ 10^5`.

In [1]:
from collections import deque
from itertools import combinations
import random, time

examples = [("110", 1, 1), ("0101", 3, 2), ("101", 2, -1)]
for s, k, exp in examples:
    print(s, k, "-> kỳ vọng", exp)

110 1 -> kỳ vọng 1
0101 3 -> kỳ vọng 2
101 2 -> kỳ vọng -1


## 3. Kiến thức nền

### 3.1. Trạng thái rút gọn

Hai chuỗi có cùng số chữ số 0 là "giống nhau" về mặt bài toán: hoán đổi tên các vị trí biến chuỗi này thành chuỗi kia, và phép thao tác (chọn `k` vị trí bất kỳ) bất biến với hoán vị vị trí. Vậy BFS trên `z ∈ [0, n]`.

### 3.2. Chuyển trạng thái là một khoảng

Chọn `x` chỉ số đang là 0 và `k − x` chỉ số đang là 1: `z' = z − x + (k − x) = z + k − 2x`. Ràng buộc: `0 ≤ x ≤ min(z, k)` và `k − x ≤ n − z` ⇒ `x ≥ max(0, k − n + z)`. Do đó `z'` chạy trên `[z + k − 2·min(z,k), z + k − 2·max(0, k−n+z)]` với bước 2.

In [2]:
def neighbors(z, n, k):
    lo = z + k - 2 * min(z, k); hi = z + k - 2 * max(0, k - n + z)
    return list(range(lo, hi + 1, 2))

print("n=4, k=3, z=2 ->", neighbors(2, 4, 3), "| z=1 ->", neighbors(1, 4, 3))

n=4, k=3, z=2 -> [1, 3] | z=1 -> [2, 4]


### 3.3. "Phần tử chưa thăm kế tiếp" bằng DSU

Mảng `nxt[v]` trỏ tới phần tử chưa thăm **cùng tính chẵn lẻ** `≥ v`. Khi thăm `v`, đặt `nxt[v] = v + 2`; `find` có nén đường. Duyệt một khoảng `[lo, hi]` chỉ chạm vào các phần tử **chưa thăm** ⇒ tổng công việc `O(n α(n))`.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

BFS từ `z₀ = s.count('0')` tới `0`. Vấn đề: mỗi đỉnh có thể có tới `k` láng giềng ⇒ BFS ngây thơ `O(n·k)`. Nhưng láng giềng là **khoảng liên tục** (cùng tính chẵn lẻ), và đỉnh nào đã thăm thì không cần xét lại ⇒ dùng con trỏ "chưa thăm kế tiếp".

Ví dụ 2 (`n = 4, k = 3`): `z₀ = 2` → láng giềng `{1, 3}`; từ `1` → `{0, 2}` ⇒ tới `0` sau 2 bước. ✔ Ví dụ 3 (`n = 3, k = 2`): `z₀ = 1` → `{1}` (lật một 0 và một 1) ⇒ không bao giờ tới 0 ⇒ `-1`. ✔

## 5. Cách 1 — Brute Force: BFS trên chuỗi

In [3]:
def brute(s, k):
    n = len(s); start = sum(1 << i for i, c in enumerate(s) if c == "1"); full = (1 << n) - 1
    masks = [sum(1 << i for i in c) for c in combinations(range(n), k)]
    dist = {start: 0}; q = deque([start])
    while q:
        u = q.popleft()
        if u == full: return dist[u]
        for m in masks:
            v = u ^ m
            if v not in dist: dist[v] = dist[u] + 1; q.append(v)
    return -1

for s, k, exp in examples:
    print(brute(s, k), "kỳ vọng", exp)

1 kỳ vọng 1
2 kỳ vọng 2
-1 kỳ vọng -1


**Nút thắt:** `2^n` trạng thái × `C(n, k)` thao tác.

## 6. Ý tưởng thuật toán / bước đột phá

**BFS + DSU nhảy:** đánh dấu `z₀` đã thăm; mỗi lớp BFS, với mỗi `cur` lấy khoảng `[lo, hi]` (mục 3.2), duyệt `v = find(lo)` trong khi `v ≤ hi`: đẩy `v` vào hàng đợi, xoá `v`, `v = find(v)`. Gặp `0` ⇒ trả số lớp.

**Đại số (Cách 4):** sau `t` thao tác, gọi `f_i` = số lần vị trí `i` bị lật. Cần: vị trí 0 bị lật **lẻ** lần, vị trí 1 **chẵn** lần, `0 ≤ f_i ≤ t`, `Σ f_i = t·k`. Ngược lại, mọi bộ `f` như vậy thực hiện được bằng `t` thao tác (mỗi thao tác chọn `k` vị trí khác nhau — điều kiện Gale–Ryser cho ma trận 0/1 với tổng hàng đều bằng `k`). Do đó `t` khả thi ⇔ `t·k ≡ z (mod 2)` và `z ≤ t·k ≤ z·A + (n − z)·B` với `A` = số lẻ lớn nhất `≤ t`, `B` = số chẵn lớn nhất `≤ t`.

## 7. Các cách tiếp theo

- **Cách 2:** BFS trên `z` duyệt **mọi** láng giềng — `O(n · k)`.
- **Cách 3:** BFS trên `z` + DSU "chưa thăm kế tiếp" — `O(n α(n))` (lời giải nộp).
- **Cách 4:** thử `t = 1, 2, …` với điều kiện đóng — `O(n)`.

### 7.1. Cách 2 — BFS ngây thơ trên số chữ số 0

In [4]:
def bfs_naive(s, k):
    n = len(s); z0 = s.count("0")
    dist = [-1] * (n + 1); dist[z0] = 0; q = deque([z0])
    while q:
        z = q.popleft()
        if z == 0: return dist[z]
        for v in neighbors(z, n, k):
            if dist[v] == -1: dist[v] = dist[z] + 1; q.append(v)
    return -1

for s, k, exp in examples:
    print(bfs_naive(s, k), "kỳ vọng", exp)

1 kỳ vọng 1
2 kỳ vọng 2
-1 kỳ vọng -1


### 7.2. Cách 3 — BFS + DSU nhảy (lời giải chính)

In [5]:
def bfs_dsu(s, k):
    n = len(s); z0 = s.count("0")
    if z0 == 0: return 0
    nxt = list(range(n + 3))                       # nxt[v]: phần tử chưa thăm cùng tính chẵn lẻ >= v

    def find(x):
        r = x
        while nxt[r] != r: r = nxt[r]
        while nxt[x] != r: nxt[x], x = r, nxt[x]
        return r

    nxt[z0] = z0 + 2
    q = deque([z0]); d = 0
    while q:
        for _ in range(len(q)):
            cur = q.popleft()
            if cur == 0: return d
            lo = cur + k - 2 * min(cur, k); hi = cur + k - 2 * max(0, k - n + cur)
            v = find(lo)
            while v <= hi:
                q.append(v); nxt[v] = v + 2; v = find(v)
        d += 1
    return -1

for s, k, exp in examples:
    print(bfs_dsu(s, k), "kỳ vọng", exp)

1 kỳ vọng 1
2 kỳ vọng 2
-1 kỳ vọng -1


### 7.3. Cách 4 — Điều kiện đóng theo số lần lật

In [6]:
def closed_form(s, k):
    n = len(s); z = s.count("0")
    if z == 0: return 0
    for t in range(1, 2 * n + 3):
        if (t * k - z) % 2: continue
        A = t if t % 2 else t - 1                  # số lần lật lẻ lớn nhất cho vị trí '0'
        B = t if t % 2 == 0 else t - 1             # số lần lật chẵn lớn nhất cho vị trí '1'
        if z <= t * k <= z * A + (n - z) * B:
            return t
    return -1

for s, k, exp in examples:
    print(closed_form(s, k), "kỳ vọng", exp)

1 kỳ vọng 1
2 kỳ vọng 2
-1 kỳ vọng -1


**Vì sao thử tới `2n + 2` là đủ?** Nếu có lời giải thì BFS trên `n + 1` trạng thái cho lời giải ngắn nhất `≤ n`. Chặn `2n + 2` an toàn và vẫn `O(n)`.

## 8. TRACE từng bước

In [7]:
def trace(s, k):
    n = len(s); z0 = s.count("0"); dist = {z0: 0}; layer = [z0]; d = 0
    print(f"n = {n}, k = {k}, số chữ số 0 ban đầu = {z0}")
    while layer:
        print(f"  lớp {d}: {layer}")
        if 0 in layer: print("  -> đến 0 sau", d, "thao tác"); return d
        new = []
        for z in layer:
            for v in neighbors(z, n, k):
                if v not in dist: dist[v] = d + 1; new.append(v)
        layer = sorted(new); d += 1
    print("  -> không tới được 0"); return -1

trace("0101", 3)
trace("0000000", 4)

n = 4, k = 3, số chữ số 0 ban đầu = 2
  lớp 0: [2]
  lớp 1: [1, 3]
  lớp 2: [0, 4]
  -> đến 0 sau 2 thao tác
n = 7, k = 4, số chữ số 0 ban đầu = 7
  lớp 0: [7]
  lớp 1: [3]
  lớp 2: [1, 5]
  -> không tới được 0


-1

In [8]:
# Bảng kiểm tra điều kiện đóng cho s có z = 5 số 0, n = 8, k = 3
n, z, k = 8, 5, 3
for t in range(1, 7):
    A = t if t % 2 else t - 1; B = t if t % 2 == 0 else t - 1
    print(f"t={t}: t·k={t*k:>2}, cùng tính chẵn lẻ với z? {((t*k - z) % 2 == 0)!s:>5}, khoảng [{z}, {z*A + (n-z)*B}] -> {'khả thi' if (t*k - z) % 2 == 0 and z <= t*k <= z*A + (n-z)*B else 'không'}")
print("BFS:", bfs_dsu("0" * 5 + "1" * 3, 3))

t=1: t·k= 3, cùng tính chẵn lẻ với z?  True, khoảng [5, 5] -> không
t=2: t·k= 6, cùng tính chẵn lẻ với z? False, khoảng [5, 11] -> không
t=3: t·k= 9, cùng tính chẵn lẻ với z?  True, khoảng [5, 21] -> khả thi
t=4: t·k=12, cùng tính chẵn lẻ với z? False, khoảng [5, 27] -> không
t=5: t·k=15, cùng tính chẵn lẻ với z?  True, khoảng [5, 37] -> khả thi
t=6: t·k=18, cùng tính chẵn lẻ với z? False, khoảng [5, 43] -> không
BFS: 3


## 9. Code nộp LeetCode

In [9]:
class Solution:
    def minOperations(self, s: str, k: int) -> int:
        n = len(s)
        z0 = s.count('0')
        if z0 == 0:
            return 0
        nxt = list(range(n + 3))

        def find(x):
            r = x
            while nxt[r] != r:
                r = nxt[r]
            while nxt[x] != r:
                nxt[x], x = r, nxt[x]
            return r

        nxt[z0] = z0 + 2
        q = deque([z0])
        d = 0
        while q:
            for _ in range(len(q)):
                cur = q.popleft()
                if cur == 0:
                    return d
                lo = cur + k - 2 * min(cur, k)
                hi = cur + k - 2 * max(0, k - n + cur)
                v = find(lo)
                while v <= hi:
                    q.append(v)
                    nxt[v] = v + 2
                    v = find(v)
            d += 1
        return -1

sol = Solution()
for s, k, exp in examples:
    got = sol.minOperations(s, k)
    print(s, k, "->", got, "✔" if got == exp else "✘")

110 1 -> 1 ✔
0101 3 -> 2 ✔
101 2 -> -1 ✔


## 10. Vì sao thuật toán đúng

1. **Rút gọn trạng thái:** tập chuỗi đạt được từ `s` sau `t` bước chỉ phụ thuộc số chữ số 0 (3.1) ⇒ khoảng cách BFS trên `z` bằng khoảng cách thật.
2. **Láng giềng:** với `x` hợp lệ trong `[max(0, k−n+z), min(z, k)]`, `z' = z + k − 2x` phủ đúng khoảng cùng tính chẵn lẻ (3.2).
3. **DSU nhảy:** mỗi đỉnh bị xoá khỏi cấu trúc khi lần đầu được thăm; `find(lo)` trả đỉnh chưa thăm nhỏ nhất cùng tính chẵn lẻ `≥ lo` (do `nxt[v] = v + 2` giữ tính chẵn lẻ). Vì vậy mỗi đỉnh vào hàng đợi đúng một lần với khoảng cách nhỏ nhất (tính chất BFS).
4. **Cách 4:** `t` thao tác khả thi ⇔ tồn tại vectơ `f` như mục 6. Đặt `f` = 1 cho mọi số 0, 0 cho mọi số 1 (tổng `z`), rồi tăng từng bước 2 ở các vị trí còn chỗ (tới `A` hoặc `B`) — đạt mọi tổng cùng tính chẵn lẻ trong `[z, zA + (n−z)B]`. Điều kiện hàng/cột: ma trận `t × n` nhị phân với mọi hàng tổng `k`, cột tổng `f_i ≤ t`, `Σ f_i = tk` luôn tồn tại (xếp "vòng tròn": điền lần lượt các cột vào các hàng theo vòng).

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. BFS chuỗi | `O(2^n · C(n,k))` | `O(2^n)` |
| 2. BFS trên `z` ngây thơ | `O(n · k)` | `O(n)` |
| 3. BFS + DSU nhảy | `O(n α(n))` | `O(n)` |
| 4. Điều kiện đóng | `O(n)` | `O(1)` |

## 12. Bộ test

In [10]:
METHODS = [brute, bfs_naive, bfs_dsu, closed_form, lambda s, k: Solution().minOperations(s, k)]
for s, k, exp in examples:
    assert all(f(s, k) == exp for f in METHODS)
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [11]:
# 12.2 ca biên
assert Solution().minOperations("1", 1) == 0
assert Solution().minOperations("0", 1) == 1
assert Solution().minOperations("00", 2) == 1
assert Solution().minOperations("01", 2) == -1            # k = n: luôn lật cả chuỗi
assert Solution().minOperations("1111", 3) == 0
print("Ca biên: OK")

Ca biên: OK


In [12]:
# 12.3 ngẫu nhiên: so với BFS chuỗi
rng = random.Random(3666)
for _ in range(2000):
    n = rng.randint(1, 8); s = "".join(rng.choice("01") for _ in range(n)); k = rng.randint(1, n)
    b = brute(s, k)
    assert all(f(s, k) == b for f in METHODS[1:]), (s, k)
print("2000 test ngẫu nhiên: OK")

2000 test ngẫu nhiên: OK


In [13]:
# 12.4 test lớn
rng = random.Random(1)
s = "".join(rng.choice("01") for _ in range(10**5))
for k in (1, 2, 3, 777, 50000, 99999, 10**5):
    t0 = time.time(); a = Solution().minOperations(s, k); t1 = time.time()
    print(f"k = {k:>6}: {a:>6} | {t1 - t0:.2f}s | khớp đóng: {a == closed_form(s, k)}")

k =      1:  50022 | 0.07s | khớp đóng: True
k =      2:  25011 | 0.04s | khớp đóng: True


k =      3:  16674 | 0.08s | khớp đóng: True
k =    777:     66 | 0.08s | khớp đóng: True


k =  50000:      2 | 0.04s | khớp đóng: True
k =  99999:  50022 | 0.07s | khớp đóng: True


k = 100000:     -1 | 0.00s | khớp đóng: True


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Tham lam: luôn lật nhiều số 0 nhất có thể

In [14]:
def wrong_greedy(s, k):
    n = len(s); z = s.count("0"); steps = 0; seen = set()
    while z and z not in seen and steps <= 2 * n:
        seen.add(z); x = min(z, k)
        if k - x > n - z: return -1
        z = z + k - 2 * x; steps += 1                               # SAI: không xét đường vòng
    return steps if z == 0 else -1

print("sai:", wrong_greedy("0101", 3), "| đúng:", brute("0101", 3))

sai: -1 | đúng: 2


### Lỗi 2 — Quên giới hạn số chữ số 1 có thể lật (`k − x ≤ n − z`)

In [15]:
def wrong_bound(s, k):
    n = len(s); z0 = s.count("0"); dist = {z0: 0}; q = deque([z0])
    while q:
        z = q.popleft()
        if z == 0: return dist[z]
        for x in range(0, min(z, k) + 1):                           # SAI: thiếu x >= k - (n - z)
            v = z + k - 2 * x
            if 0 <= v <= n and v not in dist: dist[v] = dist[z] + 1; q.append(v)
    return -1

rng = random.Random(10)
while True:
    n = rng.randint(2, 7); t = "".join(rng.choice("01") for _ in range(n)); k = rng.randint(1, n)
    if wrong_bound(t, k) != brute(t, k): break
print("phản ví dụ:", t, k, "| sai:", wrong_bound(t, k), "| đúng:", brute(t, k))

phản ví dụ: 11010 5 | sai: 2 | đúng: -1


### Lỗi 3 — DSU nhảy với bước 1 (trộn hai tính chẵn lẻ)

In [16]:
def wrong_step1(s, k):
    n = len(s); z0 = s.count("0")
    if z0 == 0: return 0
    nxt = list(range(n + 3))
    def find(x):
        while nxt[x] != x: x = nxt[x]
        return x
    nxt[z0] = z0 + 1; q = deque([z0]); d = 0                        # SAI: +1
    while q:
        for _ in range(len(q)):
            cur = q.popleft()
            if cur == 0: return d
            lo = cur + k - 2 * min(cur, k); hi = cur + k - 2 * max(0, k - n + cur)
            v = find(lo)
            while v <= hi: q.append(v); nxt[v] = v + 1; v = find(v)
        d += 1
    return -1

rng = random.Random(9)
while True:
    n = rng.randint(2, 7); s = "".join(rng.choice("01") for _ in range(n)); k = rng.randint(1, n)
    if wrong_step1(s, k) != brute(s, k): break
print("phản ví dụ:", s, k, "| sai:", wrong_step1(s, k), "| đúng:", brute(s, k))

phản ví dụ: 0101101 2 | sai: 2 | đúng: -1


### Lỗi 4 — Kết luận `-1` chỉ dựa vào tính chẵn lẻ

Điều kiện chẵn lẻ là cần nhưng chưa đủ: ví dụ `k = n` thì mỗi thao tác lật cả chuỗi, chỉ tới được toàn 1 khi chuỗi đang toàn 0 hoặc toàn 1.

In [17]:
def wrong_parity_only(s, k):
    z = s.count("0")
    if z == 0: return 0
    return -1 if (k % 2 == 0 and z % 2 == 1) else 1                 # SAI hoàn toàn

print("sai:", wrong_parity_only("0011", 4), "| đúng:", brute("0011", 4))

sai: 1 | đúng: -1


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thao tác đối xứng với hoán vị vị trí ⇒ rút gọn trạng thái thành **số lượng**.
- BFS mà mỗi đỉnh nối tới cả một **khoảng** đỉnh ⇒ "xoá khi thăm" bằng DSU nhảy / SortedList / segment tree.
- Bài "lật/chọn đúng `k` mỗi lần" ⇒ nhìn theo **số lần mỗi vị trí bị chọn** — thường cho điều kiện đóng.

## 15. Biến thể của bài

1. **Chọn tối đa `k`** (không bắt buộc đúng `k`): luôn khả thi, đáp án `⌈z / k⌉`.
2. **Biến tất cả thành '0'** thay vì '1': đổi vai trò, `z ← n − z`.
3. **Chọn `k` chỉ số liên tiếp** (LC 995): tham lam từ trái + mảng hiệu.
4. **Chi phí khác nhau mỗi vị trí**: không còn rút gọn thành số lượng.

In [18]:
# Biến thể 3: LC 995 — lật k bit liên tiếp, tham lam + mảng hiệu
def min_k_bit_flips(bits, k):
    n = len(bits); diff = [0] * (n + 1); cur = 0; ops = 0
    for i in range(n):
        cur ^= diff[i]
        if bits[i] ^ cur == 0:
            if i + k > n: return -1
            ops += 1; cur ^= 1; diff[i + k] ^= 1
    return ops

def brute_995(bits, k):
    n = len(bits); start = sum(b << i for i, b in enumerate(bits)); full = (1 << n) - 1
    masks = [((1 << k) - 1) << i for i in range(n - k + 1)]
    dist = {start: 0}; q = deque([start])
    while q:
        u = q.popleft()
        if u == full: return dist[u]
        for m in masks:
            if u ^ m not in dist: dist[u ^ m] = dist[u] + 1; q.append(u ^ m)
    return -1

rng = random.Random(15)
for _ in range(500):
    n = rng.randint(1, 9); b = [rng.randint(0, 1) for _ in range(n)]; k = rng.randint(1, n)
    assert min_k_bit_flips(b, k) == brute_995(b, k)
print("Biến thể 3 OK")

Biến thể 3 OK


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Khi nào dùng |
|---|---|---|---|
| 1. BFS chuỗi | Định nghĩa | `O(2^n …)` | Kiểm chứng |
| 2. BFS số lượng | Rút gọn trạng thái | `O(n k)` | `k` nhỏ |
| 3. BFS + DSU nhảy | Xoá khi thăm | `O(n α)` | **Lời giải chuẩn** |
| 4. Điều kiện đóng | Đếm số lần lật | `O(n)` | Chứng minh, kiểm tra nhanh |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `minOperations("000", 2)`.

**Bài 2 (tính tay).** `minOperations("00000", 3)` — liệt kê các lớp BFS.

**Bài 3 (điền TODO).** Viết `min_ops_to_zeros(s, k)` (biến tất cả thành '0').

**Bài 4 (tự viết lại).** Viết lại Cách 3 dùng hai danh sách đã sắp (mỗi tính chẵn lẻ một danh sách) và `bisect` + xoá (chấp nhận `O(n²)` xấu nhất nhưng đơn giản).

**Bài 5 (mini challenge).** Trả về luôn **dãy thao tác** (danh sách các tập `k` chỉ số) đạt tối ưu cho `n ≤ 12`.

In [19]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def min_ops_to_zeros(s, k):
    # TODO
    pass

In [20]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def construct_ops(s, k):
    # TODO: BFS trên z lưu cha, rồi hiện thực hoá từng bước bằng cách chọn x số 0 và k-x số 1
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `z = 3, n = 3, k = 2`: láng giềng của 3 là `{1}`; của 1 là `{1, 3}`. (2) Không tới được 0. (3) `-1`.

**Bài 2.** (1) `z = 5, n = 5, k = 3`: láng giềng `{2}`. (2) Từ 2: `x ∈ [0, 2]` ⇒ `{5, 3, 1}`. (3) Từ 1 hoặc 3 ⇒ `0`? Từ 3: `x = 3` ⇒ `0` ⇒ đáp án `3`.

**Bài 3.** (1) Đảo bit của `s`. (2) Gọi lời giải. (3) Kiểm chứng.

**Bài 4.** (1) Hai list `even`, `odd` chưa thăm. (2) `bisect_left(lo)` rồi cắt đoạn `[lo, hi]`. (3) Xoá đoạn đó bằng slicing.

**Bài 5.** (1) BFS lưu `parent[z']` và `x`. (2) Truy vết dãy `x`. (3) Mỗi bước chọn `x` vị trí đang là 0 và `k − x` vị trí đang là 1.

## 19. Đáp án tham khảo cho bài tập

In [21]:
# Bài 1, 2
assert Solution().minOperations("000", 2) == -1 == brute("000", 2)
assert Solution().minOperations("00000", 3) == 3 == brute("00000", 3)
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [22]:
# Bài 3, 4
from bisect import bisect_left, bisect_right

def min_ops_to_zeros_sol(s, k):
    return Solution().minOperations("".join("1" if c == "0" else "0" for c in s), k)

def bfs_sorted_lists(s, k):
    n = len(s); z0 = s.count("0")
    if z0 == 0: return 0
    rest = [[v for v in range(0, n + 1, 2) if v != z0], [v for v in range(1, n + 1, 2) if v != z0]]
    q = deque([z0]); d = 0
    while q:
        for _ in range(len(q)):
            cur = q.popleft()
            if cur == 0: return d
            lo = cur + k - 2 * min(cur, k); hi = cur + k - 2 * max(0, k - n + cur)
            arr = rest[lo % 2]; i = bisect_left(arr, lo); j = bisect_right(arr, hi)
            q.extend(arr[i:j]); del arr[i:j]
        d += 1
    return -1

rng = random.Random(4)
for _ in range(800):
    n = rng.randint(1, 8); s = "".join(rng.choice("01") for _ in range(n)); k = rng.randint(1, n)
    t = "".join("1" if c == "0" else "0" for c in s)
    assert min_ops_to_zeros_sol(s, k) == brute(t, k)
    assert bfs_sorted_lists(s, k) == brute(s, k)
print("Bài 3, 4 OK")

Bài 3, 4 OK


In [23]:
# Bài 5: dựng dãy thao tác cụ thể
def construct_ops_sol(s, k):
    n = len(s); z0 = s.count("0"); par = {z0: None}; q = deque([z0])
    while q:
        z = q.popleft()
        if z == 0: break
        for x in range(max(0, k - n + z), min(z, k) + 1):
            v = z + k - 2 * x
            if v not in par: par[v] = (z, x); q.append(v)
    if 0 not in par: return None
    xs = []; v = 0
    while par[v] is not None: z, x = par[v]; xs.append(x); v = z
    xs.reverse(); bits = [int(c) for c in s]; ops = []
    for x in xs:
        zeros = [i for i in range(n) if bits[i] == 0]; ones = [i for i in range(n) if bits[i] == 1]
        chosen = zeros[:x] + ones[:k - x]; ops.append(sorted(chosen))
        for i in chosen: bits[i] ^= 1
    assert all(bits)
    return ops

rng = random.Random(5)
for _ in range(500):
    n = rng.randint(1, 8); s = "".join(rng.choice("01") for _ in range(n)); k = rng.randint(1, n)
    ops = construct_ops_sol(s, k); b = brute(s, k)
    assert (ops is None and b == -1) or (len(ops) == b and all(len(set(o)) == k for o in ops))
print("Bài 5 OK; ví dụ 2:", construct_ops_sol("0101", 3))

Bài 5 OK; ví dụ 2: [[0, 1, 3], [1, 2, 3]]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
z0 = s.count('0'); if z0 == 0: return 0
nxt = list(range(n + 3)); nxt[z0] = z0 + 2        # DSU nhảy theo bước 2
BFS theo lớp:
    lo = cur + k - 2*min(cur, k); hi = cur + k - 2*max(0, k - n + cur)
    v = find(lo); while v <= hi: push v; nxt[v] = v + 2; v = find(v)
```

**Checklist:**

- [ ] Trạng thái là **số chữ số 0**?
- [ ] Ràng buộc `x ≥ k − (n − z)`?
- [ ] DSU nhảy **bước 2** (giữ tính chẵn lẻ)?
- [ ] `z0 = 0` ⇒ `0` ngay?
- [ ] Đã so với BFS trên chuỗi cho `n ≤ 8`?